# Insimul DSL

**Domain:** Symbolic AI & Logic  ·  **from study list**  ·  **runnable:** no — conceptual / CLI / snippets  ·  _niche/proprietary DSL_

> A self-contained refresher on **Insimul** (`github.com/danieldekerlegand/insimul`): a personal social-simulation platform that unifies three narrative-AI systems behind a single rule language — **Ensemble** (predicate / volition rules), **Kismet** (a small ASP/Prolog-style social-sim language), and **Talk of the Town** (procedural town + deep character genealogy). The name is the point: *insimul* is Latin for "at the same time / together" (also the etymological root of *ensemble*), and the engine's whole job is running all three rule systems **simultaneously** over one shared social state. This is the integration-layer sequel to [`ensemble.ipynb`](ensemble.ipynb) and [`social-physics.ipynb`](social-physics.ipynb) — skim those first for the CiF → Ensemble social-physics lineage, then come here for how the three are fused.
>
> _This notebook documents a private/personal project; the unified-syntax snippets below are **schematic** (concept-accurate, not a copy-paste grammar) — check the repo for the exact, current grammar. The three **constituent** systems are public and well-documented (real links in §8)._

## 1. What & Why

**What it is.** Insimul is an **integration engine + DSL** that lets you author one social simulation whose rules come from three different traditions and run together over a single shared world:

- **Ensemble** — *rules-based social physics*. Weighted **volition rules** score "who wants to do what to whom"; **trigger rules** keep the social state consistent; **actions** apply effects. (Full treatment in [`ensemble.ipynb`](ensemble.ipynb).)
- **Kismet** — *a small social-simulation language* (Summerville, Samuel et al.). You declare **traits, statuses, relationships, and actions** with logical preconditions; internally it compiles to **Answer Set Programming** (AnsProlog/clingo) to compute which actions are possible and how likely each is. (See [`answer-set-programming.ipynb`](answer-set-programming.ipynb).)
- **Talk of the Town (TotT)** — *procedural social world generation* (James Ryan & Michael Mateas). A Python framework that simulates an American small town day-by-day across ~200 years, producing characters with **full genealogy**, personalities, memories, beliefs, and social networks. (Powered *Bad News*.)

**The problem it solves.** Each of the three is strong at one slice of "believable social world" and weak at the others. Ensemble gives you *explainable, scored desire* but no world to put it in. Kismet gives you *clean declarative social rules* (ASP) but a small, abstract cast. TotT gives you a *rich, historied population with deep genealogy* but a fixed, hardcoded behavior model. Insimul's bet: generate the populated, historied world with **TotT**, express crisp social constraints in **Kismet/ASP**, and score moment-to-moment dramatic *desire* with **Ensemble** — all against **one** social state. The DSL exists so you don't have to hand-translate facts between three incompatible formats every tick.

**Two ways to author.** (1) **Native formats** — write rules in each system's own syntax (Ensemble JSON, Kismet's ASP/Prolog-style rules, TotT Python) and let the engine bridge them; or (2) the **unified Insimul syntax** — one rule language that compiles down to whichever backend(s) a rule needs.

**Reach for it when:** you want emergent, *explainable* social drama (Ensemble) on top of a *richly generated, genealogical* population (TotT), with *declarative* constraint logic (Kismet) — and you specifically want to mix all three rather than commit to one engine.

**Skip it when:** one system alone suffices. If you just need scored social moves, use Ensemble directly; if you just need a generated town, use TotT; if you just need a tiny declarative social sim, use Kismet. Insimul's cost is the integration surface (three runtimes, three data models, one glue layer) — only worth paying when you genuinely need the union.

## 2. Mental Model

**Insimul is a *bus* connecting three engines to one social state.** Think of it like a build system that targets three backends: you write rules once (or in three dialects), and a translation layer keeps a single source-of-truth world in sync while each engine does what it's best at.

```
                         ┌──────────────────────────────────────────┐
                         │        SHARED SOCIAL STATE (one world)     │
                         │  characters · genealogy · traits · statuses │
                         │  relationships · networks · event history   │
                         └───────────────┬────────────────────────────┘
                  facts in / effects out │  (the Insimul bus)
        ┌────────────────────────────────┼────────────────────────────────┐
        ▼                                 ▼                                 ▼
 ┌─────────────┐                 ┌─────────────────┐                ┌──────────────┐
 │  TALK OF    │  generates the  │     KISMET       │  what social   │   ENSEMBLE    │
 │  THE TOWN   │  populated,     │  (ASP / clingo)  │  actions are   │  (volition)   │
 │  (Python)   │  historied town │  declares legal  │  *possible*    │  scores how   │
 │  genealogy, │  + characters   │  actions via     │  given state   │  much each is │
 │  memory     │                 │  preconditions   │                │  *desired*    │
 └─────────────┘                 └─────────────────┘                └──────────────┘
        WORLD  ───────────────►  CONSTRAINTS  ───────────────►  DRAMA / SELECTION
```

**The slogan:** *TotT builds the stage and cast, Kismet says what moves are legal, Ensemble says which legal move actually happens.* One tick = generate/advance world → enumerate possible actions (ASP solve) → score desire (volition sum) → pick + apply effects → write back to the shared state → repeat.

It is the same "**facts → score → new facts**" loop you saw in [`social-physics.ipynb`](social-physics.ipynb), but with the three responsibilities (world, legality, desire) split across purpose-built engines instead of one. The DSL is the *adapter*: a predicate written once is projected into an Ensemble predicate, a Kismet/ASP atom, and a TotT attribute lookup.

## 3. Key Concepts

| Term | What it means in Insimul |
| --- | --- |
| **Shared social state** | The single source of truth: the cast plus their genealogy, traits, statuses, directed relationships/networks, and a timestamped **event history**. Every engine reads and writes *this*, not its own private store. |
| **The bus / bridge** | The glue that projects one fact into each backend's representation (Ensemble predicate ⇄ Kismet ASP atom ⇄ TotT object attribute) and merges effects back. The hard, valuable part of the project. |
| **Native-format rule** | A rule written in a backend's own language: **Ensemble JSON** (volition/trigger/action), **Kismet** (ASP/Prolog-style traits, statuses, actions), or **TotT Python** (generation/behavior). Run as-is by that backend. |
| **Insimul (unified) rule** | A rule in the project's own DSL that the compiler lowers to one or more backends — e.g. a single `action` whose *precondition* compiles to Kismet/ASP and whose *desire* compiles to an Ensemble volition rule. |
| **Trait / status** | Kismet's vocabulary: durable **traits** (shy, ambitious) and transient **statuses** (angry-at, indebted-to). Modeled as logical atoms; preconditions match over them. |
| **Volition** | Ensemble's contribution: the *relative, weighted* desire of an initiator (and responder) for an action — a sum of matching rules. Drives selection and accept/reject (see [`ensemble.ipynb`](ensemble.ipynb) §3). |
| **Genealogy / character generation** | TotT's contribution: characters generated with parents, ancestry, personality (e.g. Big-Five-ish), physical traits, memories, and social networks across simulated decades. |
| **Possible vs desired** | The core division of labor: **Kismet/ASP** answers *which* actions are *possible* in this state (legality); **Ensemble** answers *how much* each possible action is *wanted* (selection). |
| **Tick / step** | One simulation step: advance the world, ASP-solve the legal action set, score volitions, select, apply effects, run triggers, write back. |
| **Backend** | One of the three runtimes (TotT Python, Kismet→clingo, Ensemble JS/port). The TypeScript fork (`danieldekerlegandevolve`) re-homes parts of this on one runtime. |

## 4. Setup

**This is not Python-runnable from a notebook cell.** Insimul is a multi-runtime integration project (Python for TotT + the bridge, clingo for Kismet's ASP, a JS/TS port of Ensemble), not a `pip` library you `import` and drive here. So everything below is **real CLI / config**, not `print()` theater. The repo is personal/private; commands assume you have access to your own clone.

```bash
# Clone your platform (private repo)
git clone git@github.com:danieldekerlegand/insimul.git
cd insimul

# Python side (TotT + the bridge/engine). TotT targets Python; use a venv.
python3 -m venv .venv && source .venv/bin/activate
pip install -e .            # or: pip install -r requirements.txt

# Kismet compiles social rules to Answer Set Programming -> you need a solver.
#   macOS:  brew install clingo
#   pip:    pip install clingo
clingo --version           # sanity check the ASP backend

# (Optional) the TypeScript fork re-homes Ensemble-style scoring on Node.
#   git clone git@github.com:danieldekerlegandevolve/insimul.git insimul-ts
#   cd insimul-ts && npm install
```

The three **constituent** systems, if you want to study them in isolation first:

```bash
# Ensemble (social physics, JS)   -> see ensemble.ipynb
git clone https://github.com/ensemble-engine/ensemble.git

# Talk of the Town (procedural town + genealogy, Python)
git clone https://github.com/james-owen-ryan/talktown.git

# Kismet ships as a paper + reference implementation (ASP via clingo);
# see the papers in §8 for the grammar and the ANTLR4 front end.
```

**Authoring is data-first.** As with Ensemble, you spend most time editing rule/data files — a TotT generation config, Kismet trait/action declarations, Ensemble volition/trigger JSON, and (optionally) unified `.insimul` rules — then running the engine to tick the shared world and inspect what emerged.

## 5. Worked Examples

Conceptual walkthroughs — **not executed** (Insimul runs across Python + clingo + JS, not this kernel). The three native formats are faithful to each system; the **unified Insimul syntax is schematic** (concept-accurate sketch — verify against the repo).

### Example 1 — The same idea in all three native formats

Goal: *"A shy character is reluctant to confront someone they resent."* Each backend expresses one slice.

**(a) Talk of the Town — generate the cast + genealogy (Python).** TotT builds the world the other two reason over:

```python
# tott-style generation: a town simulated for decades, then pull a character
from talktown import Simulation
sim = Simulation()
sim.establish_setting()                 # found the town
while sim.year < sim.true_year_worldgen_ends:
    sim.simulate_one_timestep()         # day-by-day history -> births, jobs, marriages
alice = sim.random_person
print(alice.name, alice.personality.extroversion, [p.name for p in alice.parents])
# -> Alice Hewitt -0.6 ['Margaret Hewitt', 'Tom Hewitt']   # shy (low extroversion), with ancestry
```

**(b) Kismet — declare *what is legal* (ASP / Prolog-style).** Traits/statuses + an action with preconditions; this compiles to AnsProlog and is solved by clingo:

```prolog
% kismet-style: traits, a status, and a legal action with preconditions
trait(shy).            trait(confident).
status(resents).       % resents(A, B): A holds a grudge toward B

% the 'confront' action is POSSIBLE when the actor resents the target
action(confront, Actor, Target) :-
    character(Actor), character(Target), Actor != Target,
    holds(resents(Actor, Target)).
% clingo enumerates every (Actor, Target) for which 'confront' is possible
```

**(c) Ensemble — score *how much it is wanted* (volition JSON).** Among the legal confrontations, how badly does the actor want it?

```jsonc
// ensemble-style volition rule: shyness suppresses confrontation
{ "name": "shy -> reluctant to confront",
  "conditions": [ {"category":"trait","type":"shy","first":"x","value":true} ],
  "effects":    [ {"category":"action","type":"confront","first":"x","second":"y",
                   "value":true, "weight": -4} ] }   // negative push: holds back
```

Insimul's job is to keep `resents(Alice, Bob)` and `shy(Alice)` as **one** fact each, visible to (b) and (c), while TotT (a) is what put Alice, Bob, and their grudge in the world to begin with.

### Example 2 — One unified Insimul rule (schematic)

Instead of writing the precondition in Kismet and the desire in Ensemble separately, the unified DSL lets you state both in one rule; the compiler lowers each clause to the right backend:

```text
# confront.insimul  (SCHEMATIC — illustrates the lowering, not exact grammar)
action confront(actor, target):
    # --- legality  -> compiled to Kismet/ASP (clingo) ---
    requires resents(actor, target)
    requires actor != target

    # --- desire    -> compiled to Ensemble volition rules ---
    desire +6  when resents(actor, target) is strong
    desire -4  when trait(actor, shy)
    desire +3  when trait(target, confident)

    # --- effects   -> written back to the shared social state ---
    effect  status(actor, angry_at(target)) = true
    effect  network(actor, target, respect) -= 10
```

Reading it: `requires` clauses define the *possible* set (Kismet/ASP); `desire` clauses sum to a *volition* (Ensemble); `effect` clauses mutate the *shared state* (and may cascade through trigger rules). For Alice (shy, strongly resents Bob; Bob confident): desire = +6 − 4 + 3 = **+5** → she *wants* to confront, but the −4 from shyness means a slightly less-shy character would want it more. Nobody scripted the beat — TotT generated the grudge, ASP made it legal, Ensemble made it the move that fires.

### Example 3 — The multi-system tick (engine pseudo-code)

How one step advances the whole simulation — the heart of "running all three *insimul*":

```python
# PSEUDO-CODE of the Insimul engine loop
def tick(world):
    world.advance_time()                          # TotT: age the town, maybe new births/deaths

    legal = kismet.solve(world)                   # clingo: every possible (action, actor, target)
    if not legal:
        return                                    # quiet step

    scored = []
    for act in legal:                             # Ensemble: score desire for each legal move
        v = ensemble.calculate_volition(act, world)
        scored.append((v, act))

    v, best = max(scored)                         # selection: highest mutual volition
    if v > 0:
        world.apply(best.effects)                 # write effects back to the shared state
        world.run_trigger_rules()                 # restore consistency (Ensemble triggers)
        narrate(best)                             # -> "Alice finally confronts Bob about the betrayal."
```

CLI shape for a headless run (schematic):

```bash
insimul run --world town.cfg --rules confront.insimul --ticks 365 --seed 42             --trace volition   # explain why each chosen action scored highest
```

The `--trace volition` flag matters: emergence is only debuggable if the engine can decompose *why* a move won (Ensemble's rule-by-rule contribution), the same way the Ensemble Authoring Tool does (see §6).

## 6. Gotchas & Pitfalls

- **Three data models, one truth — the bridge is where bugs live.** A trait that's a boolean atom in Kismet, a numeric network in Ensemble, and an object attribute in TotT must stay *one* fact. Drift between the projections is the classic Insimul bug: an action is "legal" in ASP but its volition reads stale Ensemble state. Make the shared state authoritative and project *from* it every tick.
- **"Possible" and "desired" are different questions — don't conflate them.** Kismet/ASP decides legality; Ensemble decides selection. Putting desirability weights in preconditions (so undesirable acts become *impossible*) silently kills emergence; putting hard legality in volition weights lets illegal acts fire if the score is high enough.
- **ASP solve cost is combinatorial.** clingo enumerating every `(action, actor, target)` is fine for a small cast but explodes with TotT's hundreds of generated townsfolk. Scope the solve (candidate filtering, per-character neighborhoods) before each tick or the step time detonates.
- **TotT is abandoned-but-canonical (2014–2017), Python 2-era.** Expect to port/patch it; don't assume it runs clean on a modern interpreter. Pin versions and keep the generation step isolated behind the bridge.
- **Volition is *relative*, not absolute (Ensemble inheritance).** A +5 action only fires if nothing scores higher this tick; tuning is about *relative* weights across the whole set, and one nudge can starve unrelated behavior. (Same trap documented in [`ensemble.ipynb`](ensemble.ipynb) §6.)
- **Genealogy makes state huge and historied.** TotT characters carry memories, beliefs, and lineage; the shared state is far larger than Ensemble's classroom casts. Budget for serialization, querying, and the temptation to scan all of history every tick.
- **Debugging emergence across three engines is brutal.** When something weird happens, the cause may be a TotT-generated grudge × an ASP legality × an Ensemble weight. Without a `--trace` that attributes the outcome to specific rules in specific backends, you cannot set a breakpoint on "drama."
- **Native-format escape hatch invites divergence.** Mixing hand-written Ensemble JSON, Kismet ASP, and unified `.insimul` rules means the same concept can be expressed three ways with subtly different semantics. Pick one authoring path per concept and lower from it.
- **Two runtimes, two languages (and a TS fork).** Python ⇄ clingo ⇄ JS/TS means process boundaries, serialization, and version skew. The TypeScript fork exists partly to collapse this; know which runtime owns which fact.

## 7. When to Use vs Alternatives

| Approach | Good at | Trade-off vs Insimul |
| --- | --- | --- |
| **Insimul** (this project) | The *union*: TotT-generated genealogical world + Kismet/ASP legality + Ensemble scored desire, over one shared state | Heavy integration surface (3 runtimes/data models); personal/private; the bridge is yours to maintain |
| **Ensemble alone** | Scored, explainable, emergent social *moves*; mature engine + authoring tool | No world to populate, no genealogy, no declarative legality layer — you supply the cast and constraints yourself |
| **Kismet alone** | Clean *declarative* social rules; ASP gives "what's possible" rigor; casual-author friendly | Small/abstract casts; no rich generated world; ASP solve scales poorly to large populations |
| **Talk of the Town alone** | Deep procedural *world*: genealogy, memory, gossip, decades of history | Behavior model is fixed/hardcoded; not a tunable rule engine; abandoned codebase |
| **Versu / social practices** | Emergent drama via autonomous utility agents + shared practices | A different *whole-system* design, not a mix-and-match bus; see [`social-physics.ipynb`](social-physics.ipynb) |
| **Plain ASP / Prolog** ([`answer-set-programming.ipynb`](answer-set-programming.ipynb), [`swi-prolog.ipynb`](swi-prolog.ipynb)) | Maximum logical control over legality/derivation | You build *everything* — no volition scoring, no generation, no social vocabulary out of the box |
| **LLM-driven NPCs / generative agents** | Fluent, open-ended behavior with little up-front authoring | Expensive, non-deterministic, hard to explain/constrain; Insimul is cheap, explainable, deterministic. Increasingly *hybridized* (rules for state, LLM for surface text) |

**Rule of thumb:** choose Insimul only when you specifically want **all three layers together** — a richly generated genealogical population whose *legal* actions are declared logically and whose *chosen* actions are scored for drama. If you need just one layer, use that system directly; the integration cost is the whole price of admission, justified only by the union.

## 8. Resources

**The project**
- **Insimul — source** (personal repo; unifies the three systems + the DSL): https://github.com/danieldekerlegand/insimul
- **Insimul (TypeScript fork)** — re-homes parts on one runtime: https://github.com/danieldekerlegandevolve

**Ensemble (the volition / social-physics layer)**
- Ensemble Engine — source (JS): https://github.com/ensemble-engine/ensemble
- *The Ensemble Engine: Next-Generation Social Physics* (Samuel et al., FDG 2015): http://www.ben-samuel.com/wp-content/uploads/2015/09/FDG2015-The-Ensemble-Engine-Next-Generation-Social-Physics.pdf

**Kismet (the declarative / ASP legality layer)**
- *Kismet: A Small Social Simulation Language* (Summerville, Samuel et al.) — PDF: https://ceur-ws.org/Vol-2827/CAC-Paper_7.pdf
- Kismet paper (Casual Creators Workshop mirror): https://mkremins.github.io/casual-creators-workshop/papers/ICCC20_paper_190.pdf

**Talk of the Town (the procedural world / genealogy layer)**
- Talk of the Town — source (Python): https://github.com/james-owen-ryan/talktown
- Project page (James Ryan): https://www.jamesryan.world/talktown
- *Simulating Character Knowledge Phenomena in Talk of the Town* (Ryan & Mateas): https://www.researchgate.net/publication/335746180_Simulating_Character_Knowledge_Phenomena_in_Talk_of_the_Town

**Cross-links in this library:** [`ensemble.ipynb`](ensemble.ipynb) (volition/trigger/action model in depth) · [`social-physics.ipynb`](social-physics.ipynb) (the broad CiF → Ensemble → Versu lineage) · [`answer-set-programming.ipynb`](answer-set-programming.ipynb) (the clingo/ASP substrate Kismet compiles to) · [`swi-prolog.ipynb`](swi-prolog.ipynb) / [`datalog.ipynb`](datalog.ipynb) (the predicate/query foundations).